In [9]:
import os
import random
import json
from PIL import Image
from pathlib import Path
from collections import defaultdict
import numpy as np

# Set the base path of the dataset
base_path = Path("CURATED_MTF_DATASET")  # <-- Replace this with your actual dataset path
splits = ['train', 'test', 'val']
split_collage_limits = {'train': 10000, 'test': 1000, 'val': 1000}

# Where to save the generated collages and maps
output_dir = Path("collage_dataset_noise")
output_dir.mkdir(exist_ok=True)

# Utility function to get all image paths grouped by person
def collect_images_by_person(split_path):
    person_to_images = defaultdict(list)
    for race_dir in split_path.iterdir():
        if race_dir.is_dir() and not race_dir.name.startswith('.'):
            for gender_dir in race_dir.iterdir():
                if gender_dir.is_dir() and not gender_dir.name.startswith('.'):
                    for age_group_dir in gender_dir.iterdir():
                        if age_group_dir.is_dir() and not age_group_dir.name.startswith('.'):
                            for person_dir in age_group_dir.iterdir():
                                if person_dir.is_dir() and not person_dir.name.startswith('.'):
                                    person_id = f"{race_dir.name}/{gender_dir.name}/{age_group_dir.name}/{person_dir.name}"
                                    for img_file in person_dir.glob("*.*"):
                                        if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png'] and not img_file.name.startswith('.'):
                                            person_to_images[person_id].append(img_file)
    return person_to_images

# Add RGB noise padding to image after resizing to 512x512
def pad_with_noise(image: Image.Image, pad_fraction=1/3, resize_dim=512):
    image = image.resize((resize_dim, resize_dim))
    w, h = image.size
    pad_w = int(w * pad_fraction)
    pad_h = int(h * pad_fraction)

    new_w = w + 2 * pad_w
    new_h = h + 2 * pad_h

    # Create noise background
    noise = np.random.randint(0, 256, (new_h, new_w, 3), dtype=np.uint8)

    # Paste original image into the center
    noise_img = Image.fromarray(noise, 'RGB')
    noise_img.paste(image, (pad_w, pad_h))

    return noise_img

# Create 3x3 collage from 9 image paths using padded images
def create_collage(image_paths, save_path):
    padded_images = [pad_with_noise(Image.open(img)) for img in image_paths]
    img_width, img_height = padded_images[0].size
    collage = Image.new('RGB', (img_width * 3, img_height * 3))
    for i in range(3):
        for j in range(3):
            collage.paste(padded_images[i * 3 + j], (j * img_width, i * img_height))
    collage.save(save_path)

# Main logic
for split in splits:
    print(f"Processing {split}...")
    split_path = base_path / split
    if not split_path.exists():
        print(f"Path not found: {split_path}")
        continue

    person_to_images = collect_images_by_person(split_path)
    split_output = output_dir / split
    split_output.mkdir(parents=True, exist_ok=True)

    collage_map = defaultdict(list)
    collage_count = 0

    # Flatten all image paths
    all_images = [(p, img) for p, imgs in person_to_images.items() for img in imgs]
    if len(all_images) < 9:
        print(f"Not enough images in {split} to create a single collage.")
        continue

    for _ in range(split_collage_limits[split]):
        selected = random.choices(all_images, k=9)
        image_paths = [img for _, img in selected]
        label = f"{split}_collage_{collage_count}.jpg"
        save_path = split_output / label
        create_collage(image_paths, save_path)

        for person_id, _ in selected:
            collage_map[person_id].append(label)

        collage_count += 1

    # Save map as JSON
    map_file = split_output / f"{split}_map.json"
    with open(map_file, "w") as f:
        json.dump(collage_map, f, indent=4)

print("Collage dataset with padded noise and JSON maps created with fixed collage counts.")

Processing train...
Processing test...
Processing val...
Collage dataset with padded noise and JSON maps created with fixed collage counts.


In [11]:
import os
import random
import json
from PIL import Image
from pathlib import Path
from collections import defaultdict

# Configure this
original_root = Path("MTK")  # <- set this
output_root = Path("pairwise_faces")
output_root.mkdir(exist_ok=True)

pair_counts = {"train": 15000, "test": 2500, "val": 2500}
resize_dim = (224, 224)

def get_person_to_images(folder):
    person_to_images = defaultdict(list)
    for root, _, files in os.walk(folder):
        if not any(part.startswith('.') for part in Path(root).parts):
            person = Path(root).relative_to(folder)
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")) and not f.startswith('.'):
                    person_to_images[str(person)].append(Path(root) / f)
    return person_to_images

def create_pairwise_face_images(split):
    split_dir = original_root / split
    output_dir = output_root / split
    output_dir.mkdir(parents=True, exist_ok=True)

    person_to_images = get_person_to_images(split_dir)
    persons = list(person_to_images.keys())

    same_pairs, diff_pairs = [], []

    # positive pairs
    while len(same_pairs) < pair_counts[split] // 2:
        person = random.choice(persons)
        images = person_to_images[person]
        if len(images) >= 2:
            img1, img2 = random.sample(images, 2)
            same_pairs.append((img1, img2, 1))

    # negative pairs
    while len(diff_pairs) < pair_counts[split] // 2:
        p1, p2 = random.sample(persons, 2)
        imgs1, imgs2 = person_to_images[p1], person_to_images[p2]
        if imgs1 and imgs2:
            img1 = random.choice(imgs1)
            img2 = random.choice(imgs2)
            diff_pairs.append((img1, img2, 0))

    all_pairs = same_pairs + diff_pairs
    random.shuffle(all_pairs)

    metadata = []

    for idx, (img1_path, img2_path, label) in enumerate(all_pairs):
        try:
            img1 = Image.open(img1_path).convert("RGB").resize(resize_dim)
            img2 = Image.open(img2_path).convert("RGB").resize(resize_dim)
            concat = Image.new("RGB", (resize_dim[0] * 2, resize_dim[1]))
            concat.paste(img1, (0, 0))
            concat.paste(img2, (resize_dim[0], 0))

            filename = f"{idx:06d}_{label}.jpg"
            concat.save(output_dir / filename)
            metadata.append({"file": filename, "label": label})
        except Exception as e:
            print(f"Skipped pair {img1_path}, {img2_path} due to {e}")

    with open(output_dir / "pairs.json", "w") as f:
        json.dump(metadata, f, indent=4)

    print(f"{split}: {len(metadata)} pairs saved in {output_dir}")

# Run this for all splits
for split in ["train", "test", "val"]:
    create_pairwise_face_images(split)

train: 15000 pairs saved in pairwise_faces/train
test: 2500 pairs saved in pairwise_faces/test
val: 2500 pairs saved in pairwise_faces/val
